In [1]:
import os, sys, json, tarfile, io
import numpy as np
from PIL import Image

SHARDS_DIR   = "/home/rishabh/Downloads/umi-pipeline-training/shards"
RDT2_DIR     = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
DATASET_PATH = "/home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr"

# Check what's inside a shard
shard_files = sorted([
    f for f in os.listdir(SHARDS_DIR) 
    if f.endswith('.tar')
])

print(f"Total shards: {len(shard_files)}")
print(f"\nFirst 10 shards: {shard_files[:10]}")
print(f"Last 10 shards:  {shard_files[-10:]}")

# Peek inside first shard
print(f"\n--- Contents of first shard: {shard_files[0]} ---")
first_shard = os.path.join(SHARDS_DIR, shard_files[0])
with tarfile.open(first_shard, 'r') as tar:
    members = tar.getnames()
    print(f"Files inside: {members[:20]}")

Total shards: 740

First 10 shards: ['shard-000000.tar', 'shard-000001.tar', 'shard-000002.tar', 'shard-000003.tar', 'shard-000004.tar', 'shard-000005.tar', 'shard-000006.tar', 'shard-000007.tar', 'shard-000008.tar', 'shard-000009.tar']
Last 10 shards:  ['shard-000730.tar', 'shard-000731.tar', 'shard-000732.tar', 'shard-000733.tar', 'shard-000734.tar', 'shard-000735.tar', 'shard-000736.tar', 'shard-000737.tar', 'shard-000738.tar', 'shard-000739.tar']

--- Contents of first shard: shard-000000.tar ---
Files inside: ['000000.action.npy', '000000.action_token.npy', '000000.image.jpg', '000000.meta.json', '000001.action.npy', '000001.action_token.npy', '000001.image.jpg', '000001.meta.json', '000002.action.npy', '000002.action_token.npy', '000002.image.jpg', '000002.meta.json', '000003.action.npy', '000003.action_token.npy', '000003.image.jpg', '000003.meta.json', '000004.action.npy', '000004.action_token.npy', '000004.image.jpg', '000004.meta.json']


In [2]:
import tarfile, json, os
import numpy as np

SHARDS_DIR = "/home/rishabh/Downloads/umi-pipeline-training/shards"
TARGET_TASK = "pick up the marker and place it in the box"

shard_files = sorted([
    f for f in os.listdir(SHARDS_DIR)
    if f.endswith('.tar')
])

print(f"Total shards: {len(shard_files)}")
print(f"Target task: '{TARGET_TASK}'")
print("\nScanning shards for task labels...")
print("-" * 50)

marker_shards  = []
other_shards   = []
unknown_shards = []

for i, shard_name in enumerate(shard_files):
    shard_path = os.path.join(SHARDS_DIR, shard_name)
    
    try:
        with tarfile.open(shard_path, 'r') as tar:
            members = tar.getnames()
            
            # Look for json/metadata files
            json_files = [m for m in members if m.endswith('.json')]
            txt_files  = [m for m in members if m.endswith('.txt')]
            all_files  = members
            
            found_task = None
            
            # Try json files first
            for jf in json_files:
                try:
                    f    = tar.extractfile(jf)
                    data = json.load(f)
                    
                    if isinstance(data, dict):
                        # Check all string values
                        for k, v in data.items():
                            if isinstance(v, str) and any(
                                task_word in v.lower() 
                                for task_word in ['pick', 'marker', 'place', 'box', 'cup', 'fold', 'drawer', 'stack']
                            ):
                                found_task = v
                                break
                        
                        if found_task:
                            break
                    
                    elif isinstance(data, str):
                        found_task = data
                        break
                        
                except:
                    continue
            
            # Try txt files
            if not found_task:
                for tf in txt_files:
                    try:
                        f    = tar.extractfile(tf)
                        text = f.read().decode('utf-8').strip()
                        if text:
                            found_task = text
                            break
                    except:
                        continue
            
            # Classify shard
            if found_task:
                if TARGET_TASK.lower() in found_task.lower():
                    marker_shards.append(shard_name)
                else:
                    other_shards.append((shard_name, found_task))
            else:
                unknown_shards.append(shard_name)
    
    except Exception as e:
        unknown_shards.append(shard_name)
    
    # Progress
    if (i + 1) % 100 == 0:
        print(f"  Processed {i+1}/{len(shard_files)} shards...")
        print(f"  Marker: {len(marker_shards)}, "
              f"Other: {len(other_shards)}, "
              f"Unknown: {len(unknown_shards)}")

print("\n" + "=" * 60)
print("SHARD DISTRIBUTION:")
print("=" * 60)
print(f"  Marker task shards: {len(marker_shards)}")
print(f"  Other task shards:  {len(other_shards)}")
print(f"  Unknown shards:     {len(unknown_shards)}")

if other_shards:
    print(f"\nOther task examples:")
    for name, task in other_shards[:5]:
        print(f"  {name}: '{task}'")

if marker_shards:
    print(f"\nMarker shards (first 10): {marker_shards[:10]}")
    print(f"Marker shards (last 10):  {marker_shards[-10:]}")

Total shards: 740
Target task: 'pick up the marker and place it in the box'

Scanning shards for task labels...
--------------------------------------------------
  Processed 100/740 shards...
  Marker: 0, Other: 0, Unknown: 100
  Processed 200/740 shards...
  Marker: 0, Other: 0, Unknown: 200
  Processed 300/740 shards...
  Marker: 0, Other: 0, Unknown: 300
  Processed 400/740 shards...
  Marker: 0, Other: 0, Unknown: 400
  Processed 500/740 shards...
  Marker: 0, Other: 0, Unknown: 500
  Processed 600/740 shards...
  Marker: 0, Other: 0, Unknown: 600
  Processed 700/740 shards...
  Marker: 0, Other: 0, Unknown: 700

SHARD DISTRIBUTION:
  Marker task shards: 0
  Other task shards:  0
  Unknown shards:     740


In [3]:
import json, os, shutil

SHARDS_DIR      = "/home/rishabh/Downloads/umi-pipeline-training/shards"
FILTERED_DIR    = "/home/rishabh/Downloads/umi-pipeline-training/shards_marker_only"
RDT2_DIR        = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"

# If shard scanning worked, use marker_shards
# Otherwise use every Nth shard (assuming tasks are interleaved)
if len(marker_shards) > 0:
    print(f"✅ Using {len(marker_shards)} identified marker shards")
    selected_shards = marker_shards
else:
    print("⚠️  Could not identify tasks from shards")
    print("    Assuming tasks are evenly distributed (5 tasks)")
    print("    Taking every 5th shard starting from 0")
    
    all_shards = sorted([
        f for f in os.listdir(SHARDS_DIR) if f.endswith('.tar')
    ])
    
    # Take every 5th shard (assuming round-robin task assignment)
    selected_shards = all_shards[::5]
    print(f"    Selected {len(selected_shards)} shards")

# Create filtered directory
os.makedirs(FILTERED_DIR, exist_ok=True)

print(f"\nCreating filtered dataset in: {FILTERED_DIR}")
print(f"Copying {len(selected_shards)} shards...")

for i, shard_name in enumerate(selected_shards):
    src = os.path.join(SHARDS_DIR, shard_name)
    dst = os.path.join(FILTERED_DIR, shard_name)
    shutil.copy2(src, dst)
    
    if (i + 1) % 50 == 0:
        print(f"  Copied {i+1}/{len(selected_shards)}...")

# Copy instructions.json with ONLY marker task
marker_instructions = {"task_0": "pick up the marker and place it in the box"}
with open(os.path.join(FILTERED_DIR, 'instructions.json'), 'w') as f:
    json.dump(marker_instructions, f, indent=2)

print(f"\n✅ Filtered dataset created!")
print(f"   Location: {FILTERED_DIR}")
print(f"   Shards: {len(selected_shards)}")
print(f"   Instruction: 'pick up the marker and place it in the box'")

# Create new dataset config for RDT2
config_content = f"""name: custom/robot_dataset
type: single
shards_dir: {FILTERED_DIR}
kwargs:
  instruction_path: {FILTERED_DIR}/instructions.json
  normalizer_path: {"/home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt"}
"""

config_path = os.path.join(RDT2_DIR, "configs/datasets/marker_only_dataset.yaml")
with open(config_path, 'w') as f:
    f.write(config_content)

print(f"\n✅ New dataset config created!")
print(f"   Config: {config_path}")
print(f"   Uses: umi_normalizer_official.pt")

⚠️  Could not identify tasks from shards
    Assuming tasks are evenly distributed (5 tasks)
    Taking every 5th shard starting from 0
    Selected 148 shards

Creating filtered dataset in: /home/rishabh/Downloads/umi-pipeline-training/shards_marker_only
Copying 148 shards...
  Copied 50/148...
  Copied 100/148...

✅ Filtered dataset created!
   Location: /home/rishabh/Downloads/umi-pipeline-training/shards_marker_only
   Shards: 148
   Instruction: 'pick up the marker and place it in the box'

✅ New dataset config created!
   Config: /home/rishabh/Downloads/umi-pipeline-training/RDT2/configs/datasets/marker_only_dataset.yaml
   Uses: umi_normalizer_official.pt


In [4]:
import os, sys, json, tarfile
import numpy as np

SHARDS_DIR  = "/home/rishabh/Downloads/umi-pipeline-training/shards"
TARGET_TASK = "pick up the marker and place it in the box"

# First, peek inside meta.json of first shard
print("=" * 60)
print("Reading meta.json from first shard...")
print("=" * 60)

first_shard = os.path.join(SHARDS_DIR, "shard-000000.tar")
with tarfile.open(first_shard, 'r') as tar:
    members = tar.getnames()
    
    # Find all meta.json files
    meta_files = [m for m in members if 'meta.json' in m]
    print(f"Meta files in shard: {meta_files[:5]}")
    
    # Read first meta.json
    for meta_name in meta_files[:3]:
        f    = tar.extractfile(meta_name)
        data = json.load(f)
        print(f"\n{meta_name} contents:")
        print(json.dumps(data, indent=2))

Reading meta.json from first shard...
Meta files in shard: ['000000.meta.json', '000001.meta.json', '000002.meta.json', '000003.meta.json', '000004.meta.json']

000000.meta.json contents:
{
  "sub_task_instruction_key": "task_0",
  "episode": 0,
  "frame": 0
}

000001.meta.json contents:
{
  "sub_task_instruction_key": "task_0",
  "episode": 0,
  "frame": 1
}

000002.meta.json contents:
{
  "sub_task_instruction_key": "task_0",
  "episode": 0,
  "frame": 2
}


In [5]:
import os, json, tarfile
import numpy as np
from collections import defaultdict

SHARDS_DIR  = "/home/rishabh/Downloads/umi-pipeline-training/shards"
TARGET_TASK = "pick up the marker and place it in the box"

shard_files = sorted([
    f for f in os.listdir(SHARDS_DIR)
    if f.endswith('.tar')
])

print(f"Total shards: {len(shard_files)}")
print(f"Target task: '{TARGET_TASK}'")
print("\nScanning meta.json in each shard...")
print("-" * 60)

marker_shards  = []
other_shards   = {}   # task → list of shards
task_counts    = defaultdict(int)

for i, shard_name in enumerate(shard_files):
    shard_path = os.path.join(SHARDS_DIR, shard_name)
    
    try:
        with tarfile.open(shard_path, 'r') as tar:
            members = tar.getnames()
            
            # Get FIRST meta.json in shard
            meta_files = [m for m in members if 'meta.json' in m]
            
            if not meta_files:
                continue
            
            f    = tar.extractfile(meta_files[0])
            meta = json.load(f)
            
            # Extract task from meta - try all possible keys
            task = None
            for key in ['task', 'instruction', 'text', 
                        'task_description', 'language',
                        'task_id', 'label']:
                if key in meta:
                    task = str(meta[key])
                    break
            
            # If no direct key, search all string values
            if not task:
                for k, v in meta.items():
                    if isinstance(v, str) and len(v) > 3:
                        task = v
                        break
            
            if task:
                task_counts[task] += 1
                if TARGET_TASK.lower() in task.lower():
                    marker_shards.append(shard_name)
                else:
                    if task not in other_shards:
                        other_shards[task] = []
                    other_shards[task].append(shard_name)
                    
    except Exception as e:
        pass
    
    # Progress every 100
    if (i + 1) % 100 == 0:
        print(f"  Processed {i+1}/{len(shard_files)}...")

print("\n" + "=" * 60)
print("TASK DISTRIBUTION IN SHARDS:")
print("=" * 60)
for task, count in sorted(task_counts.items(), 
                           key=lambda x: -x[1]):
    tag = "← TARGET" if TARGET_TASK.lower() in task.lower() else ""
    print(f"  [{count:3d} shards] '{task}' {tag}")

print(f"\nMarker task shards: {len(marker_shards)}")
print(f"Other task shards:  {sum(len(v) for v in other_shards.values())}")

if marker_shards:
    print(f"\nFirst 5 marker shards: {marker_shards[:5]}")
    print(f"Last  5 marker shards: {marker_shards[-5:]}")

Total shards: 740
Target task: 'pick up the marker and place it in the box'

Scanning meta.json in each shard...
------------------------------------------------------------
  Processed 100/740...
  Processed 200/740...
  Processed 300/740...
  Processed 400/740...
  Processed 500/740...
  Processed 600/740...
  Processed 700/740...

TASK DISTRIBUTION IN SHARDS:
  [740 shards] 'task_0' 

Marker task shards: 0
Other task shards:  740


In [6]:
import shutil, json, os

SHARDS_DIR   = "/home/rishabh/Downloads/umi-pipeline-training/shards"
FILTERED_DIR = "/home/rishabh/Downloads/umi-pipeline-training/shards_marker_only"
RDT2_DIR     = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"

# Use marker_shards from Cell 2
if len(marker_shards) > 0:
    print(f"✅ Found {len(marker_shards)} marker shards from meta.json")
    selected = marker_shards
else:
    print("⚠️  meta.json didn't have task key")
    print("   Check Cell 1 output for correct key name!")
    print("   Using every 5th shard as fallback...")
    all_shards = sorted([
        f for f in os.listdir(SHARDS_DIR) if f.endswith('.tar')
    ])
    selected = all_shards[::5]
    print(f"   Selected {len(selected)} shards")

# Clean and recreate filtered dir
if os.path.exists(FILTERED_DIR):
    shutil.rmtree(FILTERED_DIR)
os.makedirs(FILTERED_DIR, exist_ok=True)

print(f"\nCopying {len(selected)} shards to: {FILTERED_DIR}")

for i, shard_name in enumerate(selected):
    src = os.path.join(SHARDS_DIR, shard_name)
    dst = os.path.join(FILTERED_DIR, shard_name)
    shutil.copy2(src, dst)
    if (i + 1) % 25 == 0:
        print(f"  Copied {i+1}/{len(selected)}...")

# Create instructions.json with ONLY marker task
instructions = {
    "task_0": "pick up the marker and place it in the box"
}
with open(os.path.join(FILTERED_DIR, 'instructions.json'), 'w') as f:
    json.dump(instructions, f, indent=2)

print(f"\n✅ Filtered dataset ready!")
print(f"   Location: {FILTERED_DIR}")
print(f"   Shards:   {len(selected)}")

# Create dataset config
config = f"""name: custom/robot_dataset
type: single
shards_dir: {FILTERED_DIR}
kwargs:
  instruction_path: {FILTERED_DIR}/instructions.json
  normalizer_path: /home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt
"""

config_path = os.path.join(
    RDT2_DIR, "configs/datasets/marker_only_dataset.yaml"
)
with open(config_path, 'w') as f:
    f.write(config)

print(f"✅ Dataset config saved: {config_path}")
print(f"   Uses: umi_normalizer_official.pt")

# Verify
print(f"\n📋 Config contents:")
print(config)

⚠️  meta.json didn't have task key
   Check Cell 1 output for correct key name!
   Using every 5th shard as fallback...
   Selected 148 shards

Copying 148 shards to: /home/rishabh/Downloads/umi-pipeline-training/shards_marker_only
  Copied 25/148...
  Copied 50/148...
  Copied 75/148...
  Copied 100/148...
  Copied 125/148...

✅ Filtered dataset ready!
   Location: /home/rishabh/Downloads/umi-pipeline-training/shards_marker_only
   Shards:   148
✅ Dataset config saved: /home/rishabh/Downloads/umi-pipeline-training/RDT2/configs/datasets/marker_only_dataset.yaml
   Uses: umi_normalizer_official.pt

📋 Config contents:
name: custom/robot_dataset
type: single
shards_dir: /home/rishabh/Downloads/umi-pipeline-training/shards_marker_only
kwargs:
  instruction_path: /home/rishabh/Downloads/umi-pipeline-training/shards_marker_only/instructions.json
  normalizer_path: /home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt

